In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d navoneel/brain-mri-images-for-brain-tumor-detection

In [ ]:
!unzip -q brain-mri-images-for-brain-tumor-detection.zip

replace brain_tumor_dataset/no/1 no.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    if dirs:
        print(root)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    '/content',
    classes=['yes','no'],
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    '/content',
    classes=['yes','no'],
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

In [ ]:
!pip install -q kaggle

from google.colab import files
files.upload()  # kaggle.json upload karo

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip -q brain-tumor-mri-dataset.zip

In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    if "Training" in root or "Testing" in root:
        print(root)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

training_set = train_datagen.flow_from_directory(
    '/content/Training',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

test_set = test_datagen.flow_from_directory(
    '/content/Testing',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

print(training_set.class_indices)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.preprocessing import image

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import seaborn as sns
import os

In [ ]:
print(os.listdir('/content'))

In [ ]:
train_path = "/content/Training"
test_path = "/content/Testing"

print(os.listdir(train_path))

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
training_set = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

test_set = test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
print(training_set.class_indices)

In [ ]:
cnn = Sequential()

cnn.add(
    Conv2D(
        32,
        3,
        activation='relu',
        input_shape=(224,224,3)
    )
)

cnn.add(MaxPooling2D(2))

cnn.add(
    Conv2D(
        64,
        3,
        activation='relu'
    )
)

cnn.add(MaxPooling2D(2))

cnn.add(
    Conv2D(
        128,
        3,
        activation='relu'
    )
)

cnn.add(MaxPooling2D(2))

cnn.add(Flatten())

cnn.add(Dense(256, activation='relu'))

cnn.add(Dropout(0.5))

cnn.add(Dense(4, activation='softmax'))

In [ ]:
cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
cnn.summary()

In [ ]:
history = cnn.fit(
    training_set,
    validation_data=test_set,
    epochs=5
)

In [ ]:
loss, accuracy = cnn.evaluate(test_set)

print("Test Accuracy =", accuracy*100)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend(['Train','Validation'])

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend(['Train','Validation'])

plt.show()

In [ ]:
predictions = cnn.predict(test_set)

predicted_classes = np.argmax(
    predictions,
    axis=1
)

true_classes = test_set.classes

In [ ]:
cm = confusion_matrix(
    true_classes,
    predicted_classes
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
print(
    classification_report(
        true_classes,
        predicted_classes
    )
)

In [ ]:
cnn.save(
    "BrainTumorCNN.h5"
)

print("Model Saved")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
img_path = "Screenshot 2026-06-30 213524.png"

img = image.load_img(
    img_path,
    target_size=(224,224)
)

img_array = image.img_to_array(img)

img_array = np.expand_dims(
    img_array,
    axis=0
)

img_array = img_array / 255.0

prediction = cnn.predict(img_array)

pred_class = np.argmax(prediction)

classes = list(training_set.class_indices.keys())

print("Predicted Class:", classes[pred_class])
print("Confidence:", round(np.max(prediction)*100,2), "%")